# Week 4 Assessment - Enhanced Sidekick

**Author: Jemine Mene-Ejegi**

Week 4 assessment, which is an enhanced version of `4_lab4.ipynb` (The Sidekick):

| Enhancement | Implementation | Source requirement |
|---|---|---|
| **Clarifying Questions** | Before any work starts, the sidekick asks 3 targeted questions to sharpen the task | *"have it so the first thing it does is ask three clarifying questions"* |
| **Planner Agent** | A dedicated planning node breaks the task into steps before the worker executes | *"have a planning agent that first decides on what tasks need to be done"* |
| **More Tools** | Added web search (Serper), push notifications (Pushover), and file write on top of Playwright | *"add on tools, add on a sequel tool... add in tools specific to things you do"* |
| **SQLite Memory** | Persistent cross-session memory via `SqliteSaver` instead of in-memory `MemorySaver` | *"change that to be the SQL memory... it will be able to remember who you are"* |
| **Persistent Sessions** | Username textbox maps to a stable thread ID so conversations resume across page reloads | *"use your username as the conversation thread"* |

In [ ]:
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langchain.agents import Tool as LangChainTool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from IPython.display import Image, display
import gradio as gr
import aiosqlite
import requests
import uuid
import os
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()

In [ ]:
load_dotenv(override=True)

openai_key    = os.getenv('OPENAI_API_KEY')
serper_key    = os.getenv('SERPER_API_KEY')
pushover_user  = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')

print(f"OpenAI key  : {'found' if openai_key else 'MISSING'}")
print(f"Serper key  : {'found (web search enabled)' if serper_key else 'not set (web search disabled)'}")
print(f"Pushover    : {'configured' if (pushover_user and pushover_token) else 'not set (push notifications disabled)'}")

## Pydantic Schemas for Structured Outputs

In [ ]:
class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(description="True if more input is needed from the user")


print("Schemas ready.")

## State

Extended from lab4 with:
- `clarifications` -- user's answers to the 3 clarifying questions
- `execution_plan` -- the planner agent's step-by-step plan

In [ ]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool
    clarifications: Optional[str]   # user's answers to the 3 clarifying questions
    execution_plan: Optional[str]   # planner's step-by-step plan


print("State defined.")

## Tools

Four tool categories:
1. **Playwright browser** -- navigate, click, extract text, get hyperlinks, etc. (from lab3/lab4)
2. **Pushover** -- send push notifications to the owner's phone
3. **Serper web search** -- Google search (requires `SERPER_API_KEY`, gracefully disabled if missing)
4. **File write** -- save results to a local file

In [ ]:
# -- Playwright browser tools --------------------------------------------------
# If you hit NotImplementedError on Windows, see the note in 3_lab3.ipynb cell 7
async_browser = create_async_playwright_browser(headless=False)
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
browser_tools = toolkit.get_tools()
print(f"Browser tools: {[t.name for t in browser_tools]}")

# -- Pushover notification tool -----------------------------------------------
PUSHOVER_URL = "https://api.pushover.net/1/messages.json"

def send_push_notification(text: str) -> str:
    """Send a push notification to the owner's phone via Pushover."""
    if not (pushover_user and pushover_token):
        return "[Push skipped: Pushover keys not configured]"
    requests.post(PUSHOVER_URL, data={"token": pushover_token, "user": pushover_user, "message": text})
    return f"Push notification sent: {text}"

tool_push = LangChainTool(
    name="send_push_notification",
    func=send_push_notification,
    description="Send a push notification to the user's phone. Use when a task completes or needs attention.",
)

# -- Serper web search --------------------------------------------------------
if serper_key:
    from langchain_community.utilities import GoogleSerperAPIWrapper
    _serper = GoogleSerperAPIWrapper()
    tool_search = LangChainTool(
        name="web_search",
        func=_serper.run,
        description="Search the web for current information using Google. Input: a search query string.",
    )
    print("Web search tool: enabled")
else:
    tool_search = None
    print("Web search tool: disabled (add SERPER_API_KEY to .env to enable)")

# -- File write tool ----------------------------------------------------------
def write_file(filename_and_content: str) -> str:
    """Write content to a file. Input must be in the format: FILENAME|||CONTENT"""
    try:
        if "|||" not in filename_and_content:
            return "Error: input must be formatted as FILENAME|||CONTENT"
        filename, content = filename_and_content.split("|||", 1)
        filename = filename.strip()
        with open(filename, "w", encoding="utf-8") as f:
            f.write(content)
        return f"File written successfully: {filename} ({len(content)} chars)"
    except Exception as e:
        return f"Error writing file: {e}"

tool_file_write = LangChainTool(
    name="write_file",
    func=write_file,
    description="Write content to a local file. Input format: FILENAME|||CONTENT (separate filename and content with |||)",
)

# -- Combine all tools --------------------------------------------------------
extra_tools = [tool_push, tool_file_write]
if tool_search:
    extra_tools.append(tool_search)

all_tools = browser_tools + extra_tools
print(f"\nTotal tools loaded: {len(all_tools)}")
print(f"  {[t.name for t in all_tools]}")

## LLM Initialisation

In [ ]:
# Worker: bound to all tools
worker_llm = ChatOpenAI(model="gpt-5.4-mini")
worker_llm_with_tools = worker_llm.bind_tools(all_tools)

# Planner: no tools needed, just structured reasoning
planner_llm = ChatOpenAI(model="gpt-5.4-mini")

# Clarifier: used outside the graph, no tools
clarify_llm = ChatOpenAI(model="gpt-5.4-mini")

# Evaluator: structured output
evaluator_llm = ChatOpenAI(model="gpt-5.4-mini")
evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)

print("All LLMs ready.")

## Helper Functions

In [ ]:
def format_conversation(messages: List[Any]) -> str:
    """Format message history into a readable string for the evaluator."""
    conversation = "Conversation history:\n\n"
    for message in messages:
        if isinstance(message, HumanMessage):
            conversation += f"User: {message.content}\n"
        elif isinstance(message, AIMessage):
            text = message.content or "[Tool use]"
            conversation += f"Assistant: {text}\n"
    return conversation


print("Helpers ready.")

## Planner Node

New in this assessment. Before the worker executes anything, the planner:
- Reviews the task, user clarifications, and success criteria
- Produces a numbered step-by-step execution plan
- Injects the plan as a message so the worker can follow it

In [ ]:
def planner(state: State) -> Dict[str, Any]:
    """Break the task into concrete steps before the worker executes."""
    # Build context: task from first human message + clarifications
    human_messages = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    task = human_messages[0].content if human_messages else ""
    clarifications = state.get("clarifications") or ""

    system_msg = (
        "You are a planning agent. Given a task, user clarifications, and success criteria, "
        "produce a clear numbered execution plan with 3-5 concrete, actionable steps. "
        "Be specific about which tools to use for each step. "
        "Available tools: navigate_browser, extract_text, click_element, get_elements, "
        "extract_hyperlinks, current_webpage, previous_webpage, "
        "web_search (if enabled), send_push_notification, write_file."
    )

    user_content = f"Task: {task}"
    if clarifications:
        user_content += f"\n\nUser clarifications:\n{clarifications}"
    user_content += f"\n\nSuccess criteria: {state['success_criteria']}"
    user_content += "\n\nCreate a numbered execution plan."

    result = planner_llm.invoke([
        SystemMessage(content=system_msg),
        HumanMessage(content=user_content),
    ])
    plan = result.content

    return {
        "messages": [AIMessage(content=f"Execution Plan:\n{plan}")],
        "execution_plan": plan,
    }


print("Planner node ready.")

## Worker Node

Same as lab4 but now aware of:
- The user's clarifications
- The execution plan from the planner

In [ ]:
def worker(state: State) -> Dict[str, Any]:
    clarifications = state.get("clarifications") or ""
    execution_plan = state.get("execution_plan") or ""

    system_message = f"""You are a helpful assistant that uses tools to complete tasks.
Keep working until you have a question for the user or the success criteria is met.

Success criteria:
{state['success_criteria']}
"""
    if clarifications:
        system_message += f"\nUser clarifications:\n{clarifications}\n"

    if execution_plan:
        system_message += f"\nExecution plan from the planner (follow this):\n{execution_plan}\n"

    system_message += """
Reply with a question for the user if you need clarification, e.g.:
  Question: please clarify whether you want a summary or a full report

Otherwise reply with your final answer.
"""

    if state.get("feedback_on_work"):
        system_message += f"""
Previously your reply was rejected. Evaluator feedback:
{state['feedback_on_work']}
Please address this feedback in your new response.
"""

    # Upsert the system message
    messages = state["messages"]
    found_system = False
    for m in messages:
        if isinstance(m, SystemMessage):
            m.content = system_message
            found_system = True
    if not found_system:
        messages = [SystemMessage(content=system_message)] + messages

    response = worker_llm_with_tools.invoke(messages)
    return {"messages": [response]}


def worker_router(state: State) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "evaluator"


print("Worker node + router ready.")

## Evaluator Node

Identical to lab4: uses structured output to judge whether the success criteria is met and whether user input is needed.

In [ ]:
def evaluator(state: State) -> State:
    last_response = state["messages"][-1].content

    system_message = (
        "You are an evaluator that determines if a task has been completed successfully by an Assistant. "
        "Assess the Assistant's last response based on the given criteria. "
        "Respond with your feedback and your decision on whether the success criteria has been met "
        "and whether more user input is needed."
    )

    user_message = f"""You are evaluating a conversation between the User and Assistant.

{format_conversation(state['messages'])}

Success criteria:
{state['success_criteria']}

Final response from the Assistant:
{last_response}

Decide if the success criteria is met and whether more user input is required.
"""

    if state.get("feedback_on_work"):
        user_message += f"\nPrior feedback you gave: {state['feedback_on_work']}\n"
        user_message += "If the Assistant is repeating mistakes, mark user_input_needed=True."

    eval_msgs = [
        SystemMessage(content=system_message),
        HumanMessage(content=user_message),
    ]
    result = evaluator_llm_with_output.invoke(eval_msgs)

    return {
        "messages": [{"role": "assistant", "content": f"Evaluator: {result.feedback}"}],
        "feedback_on_work": result.feedback,
        "success_criteria_met": result.success_criteria_met,
        "user_input_needed": result.user_input_needed,
    }


def route_based_on_evaluation(state: State) -> str:
    if state["success_criteria_met"] or state["user_input_needed"]:
        return "END"
    return "worker"


print("Evaluator node + router ready.")

## Graph

Extended from lab4 with a `planner` node inserted before `worker`:

```
START -> planner -> worker -> [tools | evaluator]
tools  -> worker
evaluator -> [worker | END]
```

In [ ]:
# Async SQLite persistent memory — required when using graph.ainvoke (async context)
db_path = "sidekick_memory.db"

async def _make_memory():
    conn = await aiosqlite.connect(db_path)
    return AsyncSqliteSaver(conn)

import asyncio
memory = asyncio.get_event_loop().run_until_complete(_make_memory())
print(f"AsyncSQLite memory: {os.path.abspath(db_path)}")

# Build graph
graph_builder = StateGraph(State)

graph_builder.add_node("planner",   planner)
graph_builder.add_node("worker",    worker)
graph_builder.add_node("tools",     ToolNode(tools=all_tools))
graph_builder.add_node("evaluator", evaluator)

graph_builder.add_edge(START, "planner")
graph_builder.add_edge("planner", "worker")
graph_builder.add_conditional_edges("worker",    worker_router,             {"tools": "tools", "evaluator": "evaluator"})
graph_builder.add_edge("tools", "worker")
graph_builder.add_conditional_edges("evaluator", route_based_on_evaluation, {"worker": "worker", "END": END})

graph = graph_builder.compile(checkpointer=memory)
print("Graph compiled successfully.")

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## Gradio UI

Three-phase interaction:
- **Phase 1** (first message): Sidekick asks 3 clarifying questions before starting work
- **Phase 2** (user answers questions): Graph runs with clarifications + planner + worker
- **Phase 3+** (follow-ups): Graph continues on the same thread

Username field maps to a stable thread ID for persistent sessions across page reloads.

In [ ]:
def make_thread_id() -> str:
    return str(uuid.uuid4())


async def process_message(message, success_criteria, history, thread, username, clarifications_stored):
    """
    Gradio callback handling three phases:
    Phase 1: Generate 3 clarifying questions (no graph call).
    Phase 2: Clarifications received; invoke graph with planner + worker.
    Phase 3+: Follow-up; continue on same graph thread.
    """
    # Persistent session: use username as thread ID if provided
    thread_id = username.strip() if (username and username.strip()) else thread
    config = {"configurable": {"thread_id": thread_id}}

    user_turns = [m for m in history if m["role"] == "user"]

    # --- Phase 1: first ever message ----------------------------------------
    if len(user_turns) == 0:
        clarify_result = clarify_llm.invoke([
            SystemMessage(content=(
                "You are a helpful assistant. The user has a task for you. "
                "Ask exactly 3 concise, targeted clarifying questions (numbered 1., 2., 3.) "
                "that will help you complete the task more effectively. "
                "Cover: specific angle/output format, any constraints or deadlines, "
                "and depth/detail level required."
            )),
            HumanMessage(content=f"Task: {message}\nSuccess criteria: {success_criteria}"),
        ])
        questions = clarify_result.content
        reply = (
            "Before I start working on this, I have a few clarifying questions:\n\n"
            + questions
            + "\n\nPlease answer all three so I can serve you most effectively!"
        )
        return (
            history + [{"role": "user", "content": message}, {"role": "assistant", "content": reply}],
            None,  # clarifications_stored not set yet
        )

    # --- Phase 2: clarifications just received ------------------------------
    if len(user_turns) == 1:
        original_task = user_turns[0]["content"]
        clarifications = message

        state = {
            "messages": original_task,
            "success_criteria": success_criteria,
            "feedback_on_work": None,
            "success_criteria_met": False,
            "user_input_needed": False,
            "clarifications": clarifications,
            "execution_plan": None,
        }
        result = await graph.ainvoke(state, config=config)
        user_msg   = {"role": "user",      "content": message}
        reply_msg  = {"role": "assistant",  "content": result["messages"][-2].content}
        eval_msg   = {"role": "assistant",  "content": result["messages"][-1].content}
        return history + [user_msg, reply_msg, eval_msg], clarifications

    # --- Phase 3+: continuing conversation ----------------------------------
    state = {
        "messages": message,
        "success_criteria": success_criteria,
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False,
        "clarifications": clarifications_stored or "",
        "execution_plan": None,
    }
    result = await graph.ainvoke(state, config=config)
    user_msg  = {"role": "user",     "content": message}
    reply_msg = {"role": "assistant", "content": result["messages"][-2].content}
    eval_msg  = {"role": "assistant", "content": result["messages"][-1].content}
    return history + [user_msg, reply_msg, eval_msg], clarifications_stored


async def reset():
    return "", "", None, make_thread_id(), None


print("Gradio callbacks ready.")

In [ ]:
with gr.Blocks(theme=gr.themes.Default(primary_hue="emerald")) as demo:
    gr.Markdown("## Enhanced Sidekick -- Personal Co-worker")
    gr.Markdown(
        "Enter a task and success criteria. "
        "The sidekick will ask 3 clarifying questions, build an execution plan, do the work, and evaluate results. "
        "Enter a username to persist your session across page reloads."
    )

    thread              = gr.State(make_thread_id())
    clarifications_stored = gr.State(None)

    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=400, type="messages")

    with gr.Row():
        username = gr.Textbox(
            show_label=True,
            label="Username (optional — for persistent sessions)",
            placeholder="e.g. jemine",
            scale=1,
        )

    with gr.Group():
        with gr.Row():
            message = gr.Textbox(
                show_label=False,
                placeholder="Your request to your sidekick",
                scale=4,
            )
        with gr.Row():
            success_criteria = gr.Textbox(
                show_label=False,
                placeholder="What are your success criteria?",
                scale=4,
            )

    with gr.Row():
        reset_button = gr.Button("Reset",  variant="stop")
        go_button    = gr.Button("Go!",    variant="primary")

    inputs  = [message, success_criteria, chatbot, thread, username, clarifications_stored]
    outputs = [chatbot, clarifications_stored]

    go_button.click(process_message, inputs, outputs)
    message.submit(process_message, inputs, outputs)
    success_criteria.submit(process_message, inputs, outputs)
    reset_button.click(reset, [], [message, success_criteria, chatbot, thread, clarifications_stored])

demo.launch()